# Step 2 of RAG: Storing Chunks in ChromaDB**RAG** (Retrieval-Augmented Generation) pipeline recap:1. Document Loading & Chunking (Step 1)2. **Storing chunks in a Vector Database** ← *this notebook*3. Retrieval (searching the vector database for relevant chunks)4. Passing retrieved chunks to an LLM to generate the final answerThis notebook is self-contained — it recreates a sample document, chunks it (recap of Step 1),and then focuses fully on **storing those chunks in ChromaDB** the right way: with embeddings,useful metadata, and stable IDs so we can retrieve, update, or trace every chunk back to its source.### What you will learn1. Recap: create and chunk a document2. Why metadata (source, chunk index, etc.) matters when storing chunks3. Setting up a ChromaDB collection with a Hugging Face embedding function4. Storing chunks with meaningful IDs and metadata5. Verifying what got stored6. Querying stored chunks (a preview of Step 3: Retrieval)7. Adding a second document to show multi-document storage

### Setup — Install Required Libraries

In [1]:
!pip install -q langchain-text-splitters chromadb sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/13

## 1. Recap — Create and Chunk a Document (Step 1)We regenerate a small sample document and split it into chunks using`RecursiveCharacterTextSplitter`, exactly as in the Step 1 notebook.

In [4]:
import os
os.makedirs("docs", exist_ok=True)

article_text = """Artificial Intelligence: An Overview
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include
understanding language, recognizing images, making decisions, and learning from experience.
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.
Deep Learning takes Machine Learning further by using neural networks with many layers. These
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understanding the meaning of a sentence.
Natural Language Processing, or NLP, is the branch of AI that focuses specifically on human
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
Vector Databases store embeddings and allow fast similarity search across millions of documents.
They are a core infrastructure piece behind modern RAG systems, chatbots, and recommendation
engines used by companies around the world today."""

with open("docs/ai_overview.txt", "w") as f:
    f.write(article_text)
print("Document created: docs/ai_overview.txt")

Document created: docs/ai_overview.txt


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(article_text)
print("Number of chunks created:", len(chunks))
for i, chunk in enumerate(chunks):
    print(f"\n--- Chunk {i} ({len(chunk)} chars) ---")
    print(chunk)

Number of chunks created: 7

--- Chunk 0 (215 chars) ---
Artificial Intelligence: An Overview
Artificial Intelligence, or AI, is the field of computer science focused on building systems
that can perform tasks which normally require human intelligence. These tasks include

--- Chunk 1 (284 chars) ---
understanding language, recognizing images, making decisions, and learning from experience.
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,

--- Chunk 2 (219 chars) ---
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.
Deep Learning takes Machine Learning further by using neural networks with many layers. These

--- Chunk 3 (258 chars) ---
layered networks can automatically discover complex patterns in large amounts of data, such as
recognizing faces in photos or understan

## 2. Why Metadata Matters When Storing ChunksWhen we store a chunk in a vector database, we should **never** store just the raw text. We shouldalso attach metadata so that later, when a chunk is retrieved, we know:| Metadata field | Why it matters ||---|---|| `source` | Which original file/document this chunk came from || `chunk_index` | The chunk's position/order within that document || `char_count` | Useful for debugging or filtering overly short/long chunks |This makes results traceable — a RAG answer can cite "this came from `ai_overview.txt`, chunk 3".

In [9]:
chunk_metadatas = [
    {
        "source": "ai_overview.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(chunks)
]

chunk_ids = [f"ai_overview_chunk_{i}" for i in range(len(chunks))]

print("Example metadata for chunk 0:", chunk_metadatas[0])
print("Example id for chunk 0:", chunk_ids[0])

Example metadata for chunk 0: {'source': 'ai_overview.txt', 'chunk_index': 0, 'char_count': 215}
Example id for chunk 0: ai_overview_chunk_0


## 3. Set Up ChromaDB with a Hugging Face Embedding FunctionWe use the free `all-MiniLM-L6-v2` model so ChromaDB automatically converts every chunk's text intoan embedding vector as we add it.

In [11]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.Client()
hf_embedder = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.create_collection(
    name="rag_chunks",
    embedding_function=hf_embedder
)

print("Collection created:", collection.name)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Collection created: rag_chunks


## 4. Store the Chunks in ChromaDBWe pass `documents` (the chunk text), `metadatas`, and `ids` all together. ChromaDB embeds eachchunk automatically using the embedding function we attached above.

In [13]:
collection.add(
    documents=chunks,
    metadatas=chunk_metadatas,
    ids=chunk_ids
)

print("Total chunks stored in ChromaDB:", collection.count())

Total chunks stored in ChromaDB: 7


## 5. Verify What Got StoredAlways double-check after a storage step — inspect a few chunks along with their metadata.

In [15]:
stored = collection.get(
    ids=chunk_ids,
    include=["documents", "metadatas"]
)

for cid, doc, meta in zip(stored["ids"], stored["documents"], stored["metadatas"]):
    print(f"ID: {cid}")
    print(f"Metadata: {meta}")
    print(f"Text: {doc[:100]}...")
    print("-" * 60)

ID: ai_overview_chunk_0
Metadata: {'char_count': 215, 'source': 'ai_overview.txt', 'chunk_index': 0}
Text: Artificial Intelligence: An Overview
Artificial Intelligence, or AI, is the field of computer scienc...
------------------------------------------------------------
ID: ai_overview_chunk_1
Metadata: {'char_count': 284, 'chunk_index': 1, 'source': 'ai_overview.txt'}
Text: understanding language, recognizing images, making decisions, and learning from experience.
Machine ...
------------------------------------------------------------
ID: ai_overview_chunk_2
Metadata: {'source': 'ai_overview.txt', 'chunk_index': 2, 'char_count': 219}
Text: and it learns the underlying pattern on its own. This approach powers everything from spam filters
t...
------------------------------------------------------------
ID: ai_overview_chunk_3
Metadata: {'source': 'ai_overview.txt', 'chunk_index': 3, 'char_count': 258}
Text: layered networks can automatically discover complex patterns in large amounts

## 6. Query the Stored Chunks (Preview of Step 3: Retrieval)Now that our chunks live in ChromaDB, we can search them by meaning. This is the foundation of thenext RAG step — retrieving the most relevant chunks for a user's question.

In [17]:
results = collection.query(
    query_texts=["How does RAG combine search with language models?"],
    n_results=2,
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta["source"]} | Chunk #{meta["chunk_index"]}")
    print(doc)
    print("-" * 60)

Distance: 0.3765 | Source: ai_overview.txt | Chunk #5
language model. Instead of relying only on what a language model memorized during training, RAG
first retrieves relevant, up-to-date information from a document collection, then passes that
information to the model so it can generate a more accurate, grounded answer.
------------------------------------------------------------
Distance: 0.4400 | Source: ai_overview.txt | Chunk #4
language. NLP powers chatbots, translation tools, and search engines. A key building block of
modern NLP is the embedding, which converts words or sentences into numeric vectors that capture
meaning.
Retrieval-Augmented Generation, or RAG, is a technique that combines a search system with a
------------------------------------------------------------


**Discussion Point:** The `distance` score tells us how relevant each chunk is (lower = morerelevant). In a full RAG system, these top chunks would now be inserted into a prompt and sent to anLLM to generate a grounded final answer — that's Step 3 and Step 4.

## 7. Adding a Second Document (Multi-Document Storage)Real RAG systems store chunks from many documents in the same collection. The `source` metadata fieldis what lets us tell them apart later.

In [19]:
second_doc_text = """Embeddings and Vector Search in Practice
Modern search engines increasingly rely on embeddings rather than simple keyword matching.
An embedding model converts a search query into the same vector space as the stored documents,
allowing the system to find results that match the meaning of the query, not just its exact words.
This is especially powerful for customer support systems, where a user might phrase a question very
differently from how it appears in a help article, yet still expect to find the right answer."""

with open("docs/embeddings_in_practice.txt", "w") as f:
    f.write(second_doc_text)

second_doc_chunks = splitter.split_text(second_doc_text)

second_doc_metadatas = [
    {
        "source": "embeddings_in_practice.txt",
        "chunk_index": i,
        "char_count": len(chunk)
    }
    for i, chunk in enumerate(second_doc_chunks)
]

second_doc_ids = [f"embeddings_in_practice_chunk_{i}" for i in range(len(second_doc_chunks))]

collection.add(
    documents=second_doc_chunks,
    metadatas=second_doc_metadatas,
    ids=second_doc_ids
)

print("Chunks from second document added:", len(second_doc_chunks))
print("Total chunks now stored in ChromaDB:", collection.count())

Chunks from second document added: 2
Total chunks now stored in ChromaDB: 9


In [20]:
# Query across BOTH documents at onceresults = collection.query(    query_texts=["how do search engines use embeddings"],    n_results=3,    include=["documents", "metadatas", "distances"])for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")    print(doc)    print("-" * 60)

**Discussion Point:** Notice the top result correctly comes from `embeddings_in_practice.txt` eventhough our collection also contains chunks from `ai_overview.txt`. This is exactly how a RAG systemsearches across an entire knowledge base to find the most relevant pieces, regardless of source.

## 8. Filtering Retrieval to a Single Source (Optional but Useful)Sometimes you only want to search within one document — combine semantic search with a metadatafilter using `where`.

In [22]:
results = collection.query(
    query_texts=["what is deep learning"],
    n_results=2,
    where={"source": "ai_overview.txt"},
    include=["documents", "metadatas", "distances"]
)

for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"Distance: {distance:.4f} | Source: {meta['source']} | Chunk #{meta['chunk_index']}")
    print(doc)
    print("-" * 60)

Distance: 0.3838 | Source: ai_overview.txt | Chunk #2
and it learns the underlying pattern on its own. This approach powers everything from spam filters
to recommendation systems.
Deep Learning takes Machine Learning further by using neural networks with many layers. These
------------------------------------------------------------
Distance: 0.4933 | Source: ai_overview.txt | Chunk #1
understanding language, recognizing images, making decisions, and learning from experience.
Machine Learning is a subset of AI where systems learn patterns from data instead of following
hardcoded rules. Instead of programming every decision by hand, we feed the system many examples,
------------------------------------------------------------


## Key Takeaways (Recap for Students)| Concept | What We Did ||---|---|| Chunk storage | Stored each chunk as a `document` in ChromaDB, not the whole file at once || Embedding function | Attached `all-MiniLM-L6-v2` so ChromaDB embeds text automatically on `add()` || Metadata | Stored `source`, `chunk_index`, `char_count` with every chunk for traceability || IDs | Used predictable ids like `ai_overview_chunk_0` so chunks can be updated/deleted later || Multi-document storage | Added a second document into the same collection || Filtered retrieval | Used `where={"source": ...}` to search within a single document only |This collection of stored chunks is now ready for **Step 3: Retrieval**, where we take a user'squestion, search this same collection, and pass the top chunks to an LLM to generate the final answer.### Practice Exercises1. Add a third document of your own and confirm `collection.count()` reflects all chunks from all three.2. Re-chunk `ai_overview.txt` with `chunk_size=150` and re-store it under new ids (e.g. `..._v2_chunk_0`)   — compare retrieval results between the two chunk sizes.3. Write a helper function `add_document(collection, splitter, text, source_name)` that handles chunking,   metadata creation, and `collection.add()` in one reusable call.

## Practice Exercises — Worked Answers

**Name:** P.Dinesh kumar reddy  
**Reg No:** 99230041247

Run the notebook setup and original cells first, then work through these answers.

### Exercise 1 — Add a third document and verify the collection count

Add another source document, split it with the same splitter, create unique IDs and metadata, and add the chunks to the existing collection. `collection.count()` should equal the total chunks stored from all three documents.

In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Ensure splitter is defined if previous cells were not run
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Example third document; replace the text with any document you choose.
third_text = '''Vector databases store vector representations of content.
They support similarity search to retrieve relevant chunks for RAG applications.'''
third_source = 'third_document.txt'

third_chunks = splitter.split_text(third_text)
third_ids = [f'{third_source}_chunk_{i}' for i in range(len(third_chunks))]
third_metadatas = [{'source': third_source, 'chunk_index': i} for i in range(len(third_chunks))]

collection.add(ids=third_ids, documents=third_chunks, metadatas=third_metadatas)
print('Chunks in third document:', len(third_chunks))
print('Total chunks in collection:', collection.count())

Chunks in third document: 1
Total chunks in collection: 10


**Verification:** `collection.count()` reports the total number of stored records/chunks in the collection. If you re-run the cell, ChromaDB may raise an error for duplicate IDs; use `upsert()` or remove the previous IDs before re-running.

### Exercise 2 — Re-chunk `ai_overview.txt` with `chunk_size=150`

Create a new splitter with chunk size 150, split the same source text again, store the new chunks under a different ID suffix, then compare retrieval results for the original and smaller chunks.

In [25]:
# Map the actual session variables to match the expected ones
ai_overview_text = article_text
ai_overview_chunks = chunks

splitter_v2 = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks_v2 = splitter_v2.split_text(ai_overview_text)
ids_v2 = [f'ai_overview_v2_chunk_{i}' for i in range(len(chunks_v2))]
metadatas_v2 = [{'source': 'ai_overview.txt', 'chunk_index': i, 'chunk_size': 150} for i in range(len(chunks_v2))]

collection.add(ids=ids_v2, documents=chunks_v2, metadatas=metadatas_v2)
print('Original chunk count:', len(ai_overview_chunks))
print('150-character chunk count:', len(chunks_v2))

# Compare using the same query in each collection/query path used by your notebook.
query_text = 'What is artificial intelligence?'
print('Original-size retrieval:')
print(collection.query(query_texts=[query_text], n_results=3, where={'source': 'ai_overview.txt'}))
print('New 150-character retrieval:')
print(collection.query(query_texts=[query_text], n_results=3, where={'chunk_size': 150}))

Original chunk count: 7
150-character chunk count: 18
Original-size retrieval:
{'ids': [['ai_overview_chunk_0', 'ai_overview_v2_chunk_0', 'ai_overview_v2_chunk_3']], 'embeddings': None, 'documents': [['Artificial Intelligence: An Overview\nArtificial Intelligence, or AI, is the field of computer science focused on building systems\nthat can perform tasks which normally require human intelligence. These tasks include', 'Artificial Intelligence: An Overview\nArtificial Intelligence, or AI, is the field of computer science focused on building systems', 'Machine Learning is a subset of AI where systems learn patterns from data instead of following']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'chunk_index': 0, 'char_count': 215, 'source': 'ai_overview.txt'}, {'chunk_index': 0, 'chunk_size': 150, 'source': 'ai_overview.txt'}, {'source': 'ai_overview.txt', 'chunk_index': 3, 'chunk_size': 150}]], 'distances': [[0.20712953805923462, 0.3391

**Expected comparison:** smaller chunks can provide more focused matches, but may lose surrounding context. Compare the returned documents and distances for the same query rather than assuming one size always performs better.

### Exercise 3 — Reusable `add_document()` helper

This helper chunks input text, builds unique IDs and metadata, and stores all chunks in one reusable call.

In [26]:
import uuid

def add_document(collection, splitter, text, source_name):
    chunks = splitter.split_text(text)
    # Unique prefix prevents collisions if the same source is added more than once.
    batch_id = uuid.uuid4().hex[:8]
    ids = [f'{source_name}_{batch_id}_chunk_{i}' for i in range(len(chunks))]
    metadatas = [
        {'source': source_name, 'chunk_index': i, 'batch_id': batch_id}
        for i in range(len(chunks))
    ]
    if chunks:
        collection.add(ids=ids, documents=chunks, metadatas=metadatas)
    return {'source': source_name, 'chunks_added': len(chunks), 'ids': ids}

# Example usage:
# result = add_document(collection, splitter, third_text, 'third_document.txt')
# print(result)
# print('Total chunks:', collection.count())

**Note:** This answer assumes the notebook uses `RecursiveCharacterTextSplitter`, `ai_overview_text`, `ai_overview_chunks`, `splitter`, and `collection` as named variables. If your earlier cells use different names, update these references to match them before running.